In [1]:
# # Cell 1: Check Python executable
# import sys
# print("Python executable:", sys.executable)

In [2]:
# # Cell 2: Check Python path
# import sys
# print("Python path:")
# for path in sys.path:
#     print(f"  {path}")

## Define Variables / Import MetaData

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from cartopy import crs as ccrs 
import cartopy.feature as cfeature
import pandas as pd
import hvplot.pandas
import xarray as xr
import hvplot.xarray
import geoviews.feature as gf

In [4]:
# Define misc variables 
# I can't get the config.py to work in jupyternotebook because it does not know where $NOBACKUP is
amer_filepath = '../../ameriflux-data/'
mic_filepath = '../intermediates/'

In [5]:
# Import site metadata csv
meta_file = amer_filepath + 'AmeriFlux-site-search-results-202410071335.tsv'
ameriflux_meta = pd.read_csv(meta_file, sep='\t')
fluxnet_meta = ameriflux_meta.loc[ameriflux_meta['AmeriFlux FLUXNET Data'] == 'Yes'] 

In [6]:
# set map proj
proj=ccrs.PlateCarree()

### 1. Check sites that aren't preprocessed or plotted

In [7]:
# Get list of files and create dataframe with truncated filenames
plots_list = !ls plots/
plotted_sites = list([filename.split('_')[0] for filename in plots_list])

In [8]:
len(plotted_sites)

1

In [9]:
# Find missing sites that are not plotted:
missing_plots = [item for item in total_sites
                if item not in plotted_sites]

missing_df = pd.DataFrame(missing_plots, columns=['Missing from Plots'])
missing_df

NameError: name 'total_sites' is not defined

In [ ]:
# What about intermediates?
intermediates_list = !ls intermediates/
processed_sites = list([filename.split('_')[0] for filename in intermediates_list])

In [ ]:
missing_processed = [item for item in total_sites
                if item not in processed_sites]
missing_df2 = pd.DataFrame(missing_processed, columns=['Missing from Intermediates'])
missing_df2

##### No longer missing! Bugs fixed

### 2. Debugging pre 2001 FLUXNET data

In [ ]:
def get_single_match(pattern):
  matches = glob.glob(pattern)
  if len(matches) == 1:
      return matches[0]
  elif len(matches) == 0:
      raise ValueError(f"No matches found for: {pattern}")
  else:
      raise ValueError(f"Multiple matches found: {matches}")

timedelta = 'DD'
micasa_var_list = ['NEE', 'NPP']

#Import list of fluxnet sites
meta_file = amer_filepath + 'AmeriFlux-site-search-results-202410071335.tsv'
ameriflux_meta = pd.read_csv(meta_file, sep='\t')
fluxnet_meta = ameriflux_meta.loc[ameriflux_meta['AmeriFlux FLUXNET Data'] == 'Yes'] #use FLUXNET only
fluxnet_list = fluxnet_meta['Site ID'].to_list()

In [ ]:
# see the dates for the erroring sites
for site_ID in fluxnet_list:
    # Open site data and access time indices
    site_file = get_single_match(amer_filepath + 'AMF_' + site_ID +
                              '_FLUXNET_SUBSET_*/AMF_' + site_ID +
                              '_FLUXNET_SUBSET_' + timedelta + '*.csv')
    fluxnet_sel = pd.read_csv(site_file)
    
    # select subset of columns + convert to datetime objects
    fluxnet_sel_dates = fluxnet_sel.loc[:,['TIMESTAMP']].copy()
    fluxnet_sel_dates['TIMESTAMP'] = pd.to_datetime(fluxnet_sel_dates['TIMESTAMP'],format='%Y%m%d')
    fluxnet_sel_dates = fluxnet_sel_dates.set_index('TIMESTAMP')
    
    # Create a list of unique dates from the site
    time = fluxnet_sel_dates.index
    dates_unique = list({dt.date() for dt in time})
    dates_unique.sort()

    
    # Extract micasa data
    path = '../micasa-data/daily-0.1deg-final/holding/'
    data_path = path + 'daily/'
    
    path_list = []
    for date in dates_unique:
        f_year = str(date.year)
        f_month = f"{date.month:02}"
        filename = 'MiCASA_v1_flux_*' + date.strftime('%Y%m%d') + '.nc4'
        try:
            get_single_match(os.path.join(data_path,f_year,f_month,filename))
        except ValueError as e:
            print(f"{site_ID} has Fluxnet data for: {dates_unique[0]} to {dates_unique[-1]}")
            break

In [ ]:
# try to skip the error for one site
site_ID = "CA-Ca1"

# Open site data and access time indices
site_file = get_single_match(amer_filepath + 'AMF_' + site_ID +
                          '_FLUXNET_SUBSET_*/AMF_' + site_ID +
                          '_FLUXNET_SUBSET_' + timedelta + '*.csv')
fluxnet_sel = pd.read_csv(site_file)

# select subset of columns + convert to datetime objects
fluxnet_sel_dates = fluxnet_sel.loc[:,['TIMESTAMP']].copy()
fluxnet_sel_dates['TIMESTAMP'] = pd.to_datetime(fluxnet_sel_dates['TIMESTAMP'],format='%Y%m%d')
fluxnet_sel_dates = fluxnet_sel_dates.set_index('TIMESTAMP')

# Create a list of unique dates from the site
time = fluxnet_sel_dates.index
dates_unique = list({dt.date() for dt in time})
dates_unique.sort()


# Extract micasa data
data_path = '/discover/nobackup/hzafar/ghgc/micasa/micasa-data/daily'
path_list = []
for date in dates_unique:
    f_year = str(date.year)
    f_month = f"{date.month:02}"
    filename = 'MiCASA_v1_flux_*' + date.strftime('%Y%m%d') + '.nc4'
    # print(os.path.join(data_path,f_year,f_month,filename)
    
    try:
        filepath = get_single_match(os.path.join(data_path,f_year,f_month,filename))
        path_list.append(filepath)
    except ValueError as e:
        continue

In [ ]:
print(dates_unique[0])
path_list[0]

In [ ]:
import seaborn as sns

### FluxNet Land Type / Tables of Site lat/lon for Brad

In [ ]:
summary_table = fluxnet_meta[['Site ID','Latitude (degrees)','Longitude (degrees)', 'Vegetation Abbreviation (IGBP)', 'Vegetation Description (IGBP)', 'Climate Class Abbreviation (Koeppen)', 'Climate Class Description (Koeppen)']]
summary_table

### Plot AmeriFlux sites

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6), subplot_kw= {'projection': proj})
ax.add_feature(cfeature.COASTLINE,zorder=0)
sns.scatterplot(x='Longitude (degrees)', y='Latitude (degrees)', data=summary_table, hue='Vegetation Abbreviation (IGBP)', ax=ax)
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)

In [ ]:
# Plot vegetation abbreviation?
veg_dict = dict(zip(summary_table['Vegetation Abbreviation (IGBP)'].unique(),summary_table['Vegetation Description (IGBP)'].unique()))

In [ ]:
pprint.pprint(veg_dict)

### MiCASA Land Mask

In [ ]:
!ls ../../landmask

In [ ]:
# Import landmask file for 2001
landmask_filepath = '../../landmask/'
year = str(2001)
ds = xr.open_dataset(landmask_filepath + 'MiCASA_v1_cover_x3600_y1800_yearly_' + year + '.nc4')

In [ ]:
ds.ftype

In [ ]:
ds_water = ds.ftype.sel(type=17)

In [ ]:
ds_water.plot()

In [ ]:
# ds.ftype.hvplot(x='lat',y='lon', 
#                 crs=proj,
#                size=150)

In [ ]:
site_ID = 'US-KS3' # example site that is showing up weird
# Extract site lat/lon
site_lat = ameriflux_meta.loc[ameriflux_meta['Site ID'] == site_ID, 'Latitude (degrees)'].values[0]
site_lon = ameriflux_meta.loc[ameriflux_meta['Site ID'] == site_ID, 'Longitude (degrees)'].values[0]
print(site_lat, site_lon)

In [ ]:
# Approx location of site
ax = plt.subplot(projection=proj,frameon=False)
if site_lat >= 20:
    # North America extents
    min_lon, max_lon = -170, -57
    min_lat, max_lat = 25, 74

else:
    # South America extents
    min_lon, max_lon = -90, -30
    min_lat, max_lat = -60, 12

ax.axis('off')
ax.set_extent([min_lon, max_lon, min_lat, max_lat], crs=ccrs.PlateCarree())
ax.coastlines()

ax.scatter(site_lon,site_lat,
       marker='*', 
       s=500,
       color='yellow',
       edgecolor='black',
               zorder=3)

In [ ]:
# Subset data for plotting
min_lon, max_lon = site_lon-5, site_lon+5
min_lat, max_lat = site_lat-2, site_lat+2

#### Single file

In [ ]:
mult_path_test = glob.glob('/discover/nobackup/hzafar/ghgc/micasa/micasa-data/daily/2016/01/MiCASA_v1_flux*.nc4')

In [ ]:
ds = xr.open_dataset(mult_path_test[0])['NEE']
ds_subset = ds.sel(lat=slice(min_lat, max_lat), lon=slice(min_lon,max_lon)).isel(time=0)
ds_subset

#### Multifile

In [ ]:
import h5netcdf
import dask
dask.config.set({'array.slicing.split_large_chunks': True})

In [ ]:
mult_ds = xr.open_mfdataset(
    mult_path_test, 
    engine='h5netcdf',
    parallel=True,  # Enable parallel reading
    chunks='auto'   # Let dask choose chunk sizes
)['NEE']
mult_ds

In [ ]:
ds_subset = mult_ds.sel(lat=slice(min_lat, max_lat), lon=slice(min_lon,max_lon))
ds_subset.min().load(), ds_subset.max().load()

In [ ]:
mesh_plot = ds_subset.hvplot(x="lon", y="lat",
                      cmap='RdBu_r',
                  clim=(-2e-9,3e-8),
                      # crs = proj,
                      # rasterize=True,
                 frame_width = 500,
                 # widget_location='bottom'
                     )
mesh_plot

In [ ]:
ds_sel = ds_subset.sel(lon=[site_lon], lat=[site_lat], method='nearest')

In [ ]:
point = ds_sel.hvplot.points('lon', 'lat',
                             color='yellow',size=150,
                              # crs=proj,
                              # geo=True
                             # widget_location='bottom'
                            )
type(point)

In [ ]:
type(mesh_plot), type(point)

In [ ]:
mesh_plot * point

In [ ]:
## Let's look at some of the other sites that plot zero, where they are:
ID_list = ['US-EDN' , 'US-HB1', 'US-KS3']# example site that is showing up weird
filtered_df = fluxnet_meta[fluxnet_meta['Site ID'].isin(ID_list)]
filtered_df[['Site ID','Latitude (degrees)','Longitude (degrees)',]]

## RSME plotting

In [ ]:
# Create df with lat/lons
site_subset = ['Site ID', 
               'Longitude (degrees)',
                'Latitude (degrees)',
               ]
df_meta = fluxnet_meta[site_subset]
df_meta.set_index('Site ID')

In [ ]:
# Import and merge results
results = pd.read_csv('../analysis/results.csv',index_col='SiteID')
results

In [ ]:
df = df_meta.join(results, on='Site ID')
df

In [ ]:
ds = xr.Dataset(
    coords={
        'site_id': df['Site ID'].values,
        'lat': ('site_id', df['Latitude (degrees)'].values),
        'lon': ('site_id', df['Longitude (degrees)'].values),
    }, 
    data_vars={
        'NEE_RSME': ("site_id", df['NEE_RSME'].values),
        'NPP_RSME': ("site_id", df['NPP_RSME'].values),
    }
)

In [ ]:
ds

### Xarray matplotlib

In [ ]:
from matplotlib.colors import ListedColormap

In [ ]:
# Make transparency colormap:
cmap = plt.cm.autumn_r
cmap

In [ ]:
cmap(np.arange(cmap.N)).shape

In [ ]:
cmap(1)

In [ ]:
my_cmap = cmap(np.arange(cmap.N))
my_cmap[:, -1] = np.linspace(0, 1, cmap.N)
my_cmap[-1]

In [ ]:
my_cmap = ListedColormap(my_cmap)
my_cmap

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(12, 10), subplot_kw={'projection': proj}, constrained_layout=True);
fig.suptitle('MiCASA, FluxNet Sites Root Mean Squared Error (RSME)', y=0.76)
values =['NEE_RSME', 'NPP_RSME']
for ax,val in zip(axs, values):
    ax.add_feature(cfeature.COASTLINE,zorder=0)
    plot = ds.plot.scatter(x="lon", y="lat",ax=ax,
                           
                           markersize=val, edgecolor='none',add_legend=False,
                           
                            norm=colors.LogNorm(), 
                            # norm=colors.LogNorm(vmin=ds[val].min(), vmax=ds[val].max()),
                           hue=val,
                           cmap=my_cmap,
                           # cmap='cool',
                           # cmap='autumn_r',
                           add_colorbar=False
                          )
    
    cbar = fig.colorbar(plot, ax=ax, shrink=0.9, label=val[4:], orientation='horizontal')
    ax.set_title(val[:3])
plt.show()

#### Try ipympl? It doesn't seem to work

In [ ]:
import ipympl

In [ ]:
%matplotlib ipympl
ds.plot.scatter(x="lon", y="lat")

In [ ]:
# !jupyter labextension list

In [ ]:
%matplotlib inline

#### Xarray bokeh plot? This doesn't work so I have to plot the dataframe*

In [ ]:
# ds_NEE = ds[values[0]]
# ds_NEE

In [ ]:
# hv.extension('bokeh', inline=True)
# ds_NEE.hvplot.points(x='lon', y='lat',
#                       geo=True,
#                      # crs=proj, 
#                     # project=True
#                      )

# ds_dropped = ds_NEE.drop_indexes("site_id")
# ds_dropped = ds_dropped.drop_vars("site_id")
# ds_dropped

### Pandas Holoviews

In [ ]:
df.head()

In [ ]:
import xyzservices.providers as xyz
from matplotlib.ticker import LogFormatter

In [ ]:
min_lon, max_lon = df["Longitude (degrees)"].min(), df["Longitude (degrees)"].max()
min_lat, max_lat = df["Latitude (degrees)"].min(), df["Latitude (degrees)"].max()

print(min_lon, max_lon)
print(min_lat, max_lat)

In [ ]:
my_cmap

In [ ]:
plot = df.hvplot.points(x="Longitude (degrees)", 
                        y="Latitude (degrees)",
                        geo=True, 
                        crs=ccrs.PlateCarree(),
                        # projection=ccrs.PlateCarree(), # Doesn't work with tiles

                         #Custom cmap with transparency won't show up in bokeh
                        c=values[0],
                        logz=True,
                        cmap=my_cmap,
                        clabel=f'{values[0]}',

                         size=45,
                         # Size values don't scale logarithmically
                        # s=values[0],
                        # scale=4500,
                         # color='red',
                        
                        tiles=True,
                        tiles_opts={'alpha': 0.4},
                        # tiles=xyz.Esri.WorldGrayCanvas,


                        hover_cols=['Site ID'],

                        # width=700, height=500,
                        # xlim=(-165, -40),   # longitude range
                        # ylim=(-60, 75),     # latitude range
                        frame_width=800,
                        frame_height=500
                                           )
plot

In [ ]:
# Try to scale size by log:
# df_scale = df.copy()
# df_scale["log_NEE_RSME"] = np.log(df_scale["NEE_RSME"])
# df_scale.head()

# **** This doesn't work because the logs are negative- would have to create a pseudo log scale but this is complex ******

In [ ]:
plot * gf.coastline

## Run a random example

In [ ]:
import numpy as np
import xarray as xr

lat = np.linspace(-90, 90, 5)
lon = np.linspace(-180, 180, 5)
data = np.random.rand(len(lat), len(lon))

ds_ex = xr.DataArray(data, coords=[('lat', lat), ('lon', lon)], name='val')

ds_ex.hvplot.image(x='lon', y='lat')  # no geo=True yet

In [ ]:
ds_ex

In [ ]:
ds_ex.hvplot.points(x="lon", y="lat", 
                 # c=val,
                    geo=True,
                  projection=proj,
                 # coastline=True,
                 )

## Try geopandas?

In [ ]:
# import geopandas as gpd
# from shapely.geometry import Point


In [ ]:
# gdf = gpd.GeoDataFrame(
#     {"site_id": ds["site_id"].values},
#     geometry=[Point(xy) for xy in zip(ds["lon"].values, ds["lat"].values)],
#     crs="EPSG:4326"
# )